In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from classifier_weather.src.dataset import get_transforms, TestDataset, build_idx_to_target
from classifier_weather.src.train import train, val, train_with_kfold
from classifier_weather.src.model import create_model
from classifier_weather.src.predict import save_predict, predict_ensemble

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
TRAIN_DATA_DIR = '../data/train/train'
TEST_DATA_DIR = '../data/test/test'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
full_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

base_dataset = datasets.ImageFolder(TRAIN_DATA_DIR, transform=full_transform)

In [ ]:
data_transforms = get_transforms(IMAGENET_MEAN, IMAGENET_STD)

In [ ]:
fold_scores = train_with_kfold(TRAIN_DATA_DIR, data_transforms, device)

In [ ]:
test_dataset = TestDataset(TEST_DATA_DIR, transform=data_transforms['val'])

test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
import importlib
from classifier_weather.src import predict

importlib.reload(predict)
from classifier_weather.src.predict import predict_ensemble

model_paths = [f"../models/model_fold_{i}.pth" for i in range(5)]

filenames, final_preds = predict_ensemble(model_paths, test_dataloader, device=device)

In [ ]:
save_predict(base_dataset, final_preds, filenames, output_path='../results/kfold_ensemble_pred.csv')